# 100 pandas x polars puzzles

Inspired by [100 Pandas Puzzles](https://github.com/ajcr/100-pandas-puzzles) by [Alex Riley](https://www.linkedin.com/in/ajcriley/), here are  **100\* short data manipulation puzzles**, solved **side-by-side in Pandas and Polars**.

This is for people who are familiar with pandas and want to adopt polars to reap the performance and simplicity benefits that it offers.

## Why should you even care about polars ?
The polars philosopy can be found in the [user guide](https://docs.pola.rs/). Below i have added some notes on how these philosopy compare with pandas for context: 
- Utilizes all available cores on your machine.
    - pandas mostly uses a single core from your CPU, Polars is **designed for parallel execution** and can utilize multiple CPU cores automatically for many operations.    
- Optimizes queries to reduce unneeded work/memory allocations.
    - pandas operations are eagerly evaluated, meaning each operation materializes immediately leading to repeated memory allocation and intermediate DataFrames. 
    - Polars supports a **LazyFrame** API, where transformations like:
    `.select()`, `.filter()`, `.with_columns()`, `.group_by()`, `.join()`
    build a **query plan** instead of executing immediately.
    - Execution is triggered only when an **action** such as `.collect()` or `.fetch()` is called.
    - Before execution, Polars optimizes the entire plan (projection pushdown, predicate pushdown, reordering, parallelization). 
- Handles datasets much larger than your available RAM.
    - Polars can process datasets **larger than available RAM** in many scenarios by streaming data and avoiding unnecessary materialization.
- A consistent and predictable API.
    - Polars emphasizes **explicit, expression-based transformations**.
    - Also, Polars syntax is similar to PySpark. So if you later go down the spark route, understanding polars will be helpful
- Adheres to a strict schema (data-types should be known before running the query).
    - Polars enforces known data types before query execution.  This reduces data type surprises and bugs

PS: Polars offers both Eager (regular dataframe, but faster processes) and  Lazy execution (uses lazyframe which is even faster !!) approach. Since these puzzles are designed for interactive learning where we inspect results at every step, we will primarily use the Eager (DataFrame) API, though many solutions would be written in an identical manner in Lazy mode. [Read more about polars Lazyframe and dataframe](https://stuffbyyuki.com/lazyframe-vs-dataframe-in-polars-performance-comparison/)

Every polars solution in this repo includes a Quick Info about the logic used. Polars follows a **functional, expression-based model** built around three core concepts:
1. Constructors `(e.g., pl.col(), pl.lit())`
2. Expressions `(e.g., .sum(), .is_between(), .replace())`
3. Contexts `(e.g., .select(), .filter(), .with_columns())`

Most work in Polars happens by combining **Expressions inside a Context**.  
This allows Polars to analyze and optimize the full expression chain **before any data is processed** — one of the key reasons it performs so well.

The exercises are loosely divided in sections. Each section has a difficulty rating; these ratings are subjective, of course, but should be a seen as a rough guide as to how inventive the required solution is.

Enjoy the puzzles!

\* *the list of exercises is not yet complete! Pull requests or suggestions for additional exercises, corrections and improvements are welcomed.*

## Importing pandas / polars

Difficulty: *easy* 

##### Before you begin

These first puzzles focus on **environment awareness**, not data manipulation.

You’ll be:
- importing Pandas and Polars
- inspecting their versions
- and printing detailed environment information

At first glance, this may seem trivial. In real-world data work, it isn’t.

Understanding how to identify **what version of a library you’re running**, and **what that library depends on**, is often the first step in:
- debugging unexpected behavior
- reproducing results
- collaborating with others
- and comparing how different tools behave under the same setup

Both Pandas and Polars expose similar APIs here.  


Think of these puzzles as learning how to **check your tools before using them**.


**1.a** Import pandas under the alias `pd`.<br>
**1.b** Import polars under the alias `pl`.

## Puzzle 1
**1.a** Import pandas under the alias `pd`.<br>
**1.b** Import polars under the alias `pl`.

In [ ]:
# import pandas here
import pandas as pd

In [ ]:
# import polars here
import polars as pl

## Puzzle 2
Print the version of pandas and polars that has been imported.

In [ ]:
# print the pandas version here 
print(pd.__version__)

2.3.3


In [ ]:
# print the polars version here 
print(pl.__version__)

1.36.1


## Puzzle 3

Print out all the *version* information of the libraries that are required by pandas and polars.

In [ ]:
#pandas required libraries here
pd.show_versions()


INSTALLED VERSIONS
------------------
commit                : 9c8bc3e55188c8aff37207a74f1dd144980b8874
python                : 3.11.9
python-bits           : 64
OS                    : Windows
OS-release            : 10
Version               : 10.0.22000
machine               : AMD64
processor             : Intel64 Family 6 Model 142 Stepping 9, GenuineIntel
byteorder             : little
LC_ALL                : None
LANG                  : None
LOCALE                : English_United Kingdom.1252

pandas                : 2.3.3
numpy                 : 2.4.0
pytz                  : 2025.2
dateutil              : 2.9.0.post0
pip                   : 24.0
Cython                : None
sphinx                : None
IPython               : 9.8.0
adbc-driver-postgresql: None
adbc-driver-sqlite    : None
bs4                   : None
blosc                 : None
bottleneck            : None
dataframe-api-compat  : None
fastparquet           : None
fsspec                : None
html5lib            

In [ ]:
#polars required libraries here
pl.show_versions()

--------Version info---------
Polars:              1.36.1
Index type:          UInt32
Platform:            Windows-10-10.0.22000-SP0
Python:              3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Runtime:             rt32

----Optional dependencies----
Azure CLI            <not installed>
adbc_driver_manager  <not installed>
altair               <not installed>
azure.identity       <not installed>
boto3                <not installed>
cloudpickle          <not installed>
connectorx           <not installed>
deltalake            <not installed>
fastexcel            <not installed>
fsspec               <not installed>
gevent               <not installed>
google.auth          <not installed>
great_tables         <not installed>
matplotlib           <not installed>
numpy                2.4.0
openpyxl             <not installed>
pandas               2.3.3
polars_cloud         <not installed>
pyarrow              <not installed>
pydantic             <not

> Both Pandas and Polars expose `__version__` and `show_versions()` as part of common Python library conventions. 

> `__version__` follows a long-standing Python convention for identifying library releases.  

> `show_versions()` provides a snapshot of the runtime environment, which is essential for debugging data-related issues in real-world systems. ***basically, with `show_versions()` you will see both the pandas/polars version + a full list of its optional dependencies***

## Puzzle 4
Create a DataFrame `df` from this dictionary `data` which has the index `labels`.

## DataFrame basics

### A few of the fundamental routines for selecting, sorting, adding and aggregating data in DataFrames

Difficulty: *easy*

Before solving the puzzles below, remember to import NumPy (used here only for properly representing the missing values in the numeric column):
```python
import numpy as np
```
You will be working with the same underlying dataset for both Pandas and Polars puzzles.

Consider the following Python dictionary and list of row labels:


``` python
data = {'animal': ['cat', 'cat', 'snake', 'dog', 'dog', 'cat', 'snake', 'cat', 'dog', 'dog'],
        'age': [2.5, 3, 0.5, np.nan, 5, 2, 4.5, np.nan, 7, 3],
        'visits': [1, 3, 2, 3, 2, 3, 1, 1, 2, 1],
        'priority': ['yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no']}

labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']
```
(This is just some  made up data with the theme of animals and trips to a vet.)

> Important note: Pandas has a first-class Index that is separate from the data columns. Polars does not maintain a persistent row index; if row labels are needed, they must be represented explicitly as a column.. This difference will matter in several puzzles below.

> ***A useful mental model is that Pandas behaves more like a spreadsheet—where rows have persistent labels, while Polars behaves more like a database or query engine, where data is primarily treated as a collection of columns and rows have no inherent identity unless explicitly added..*** - Gemini , last night

> Not having to manage a global index removes the need for row-level alignment and bookkeeping. This makes it easier for Polars to reorder, chunk, and parallelize operations, contributing to its performance.




In [126]:
import numpy as np

data = {'animal': ['cat', 'cat', 'snake', 'dog', 'dog', 'cat', 'snake', 'cat', 'dog', 'dog'],
        'age': [2.5, 3, 0.5, np.nan, 5, 2, 4.5, np.nan, 7, 3],
        'visits': [1, 3, 2, 3, 2, 3, 1, 1, 2, 1],
        'priority': ['yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no']}

labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']

df = pd.DataFrame(data,index=labels)

print(df)

  animal  age  visits priority
a    cat  2.5       1      yes
b    cat  3.0       3      yes
c  snake  0.5       2       no
d    dog  NaN       3      yes
e    dog  5.0       2       no
f    cat  2.0       3       no
g  snake  4.5       1       no
h    cat  NaN       1      yes
i    dog  7.0       2       no
j    dog  3.0       1       no


In [127]:
pldf = (
    pl.DataFrame(data)
    .with_columns(pl.Series("index", labels)) # Add the index column
    .select(["index", pl.all().exclude("index")]) # Move 'index' to the left side for sanity's sake
)

print(pldf)

shape: (10, 5)
┌───────┬────────┬─────┬────────┬──────────┐
│ index ┆ animal ┆ age ┆ visits ┆ priority │
│ ---   ┆ ---    ┆ --- ┆ ---    ┆ ---      │
│ str   ┆ str    ┆ f64 ┆ i64    ┆ str      │
╞═══════╪════════╪═════╪════════╪══════════╡
│ a     ┆ cat    ┆ 2.5 ┆ 1      ┆ yes      │
│ b     ┆ cat    ┆ 3.0 ┆ 3      ┆ yes      │
│ c     ┆ snake  ┆ 0.5 ┆ 2      ┆ no       │
│ d     ┆ dog    ┆ NaN ┆ 3      ┆ yes      │
│ e     ┆ dog    ┆ 5.0 ┆ 2      ┆ no       │
│ f     ┆ cat    ┆ 2.0 ┆ 3      ┆ no       │
│ g     ┆ snake  ┆ 4.5 ┆ 1      ┆ no       │
│ h     ┆ cat    ┆ NaN ┆ 1      ┆ yes      │
│ i     ┆ dog    ┆ 7.0 ┆ 2      ┆ no       │
│ j     ┆ dog    ┆ 3.0 ┆ 1      ┆ no       │
└───────┴────────┴─────┴────────┴──────────┘


## Puzzle  5 
Display a summary of the basic information about this DataFrame and its data (*hint: there is a single method that can be called on the DataFrame*).

In [128]:
# pandas
df.describe()

,age,visits
count,8.000000,10.000000
mean,3.437500,1.900000
std,2.007797,0.875595
min,0.500000,1.000000
25%,2.375000,1.000000
50%,3.000000,2.000000
75%,4.625000,2.750000
max,7.000000,3.000000


In [129]:
#polars

pldf.describe()

statistic,index,animal,age,visits,priority
str,str,str,f64,f64,str
"""count""","""10""","""10""",10.0,10.0,"""10"""
"""null_count""","""0""","""0""",0.0,0.0,"""0"""
"""mean""",null,null,NaN,1.9,null
"""std""",null,null,NaN,0.875595,null
"""min""","""a""","""cat""",0.5,1.0,"""no"""
"""25%""",null,null,2.5,1.0,null
"""50%""",null,null,4.5,2.0,null
"""75%""",null,null,7.0,3.0,null
"""max""","""j""","""snake""",7.0,3.0,"""yes"""


In [130]:
pldf.null_count()

index,animal,age,visits,priority
u32,u32,u32,u32,u32
0,0,0,0,0


> the `.info()` method [does not exist](https://github.com/pola-rs/polars/issues/2429) for the polars dataframe 

> If you want a polars equivalent to pandas' `df.info()` use `df.null_count()`

> Note that we had 2 `np.nan` in the age column from the dictionary that was used to create our dataframes, but when you look at the result of `describe()` (or `null_count()`) for the polars dataframe there were no null values detected... 

> This is because polars actually doesn't recognize numpy's `np.nan` as null, instead it sees it as an 'unknown float' 

> But, it instead recognizes `None` as null. 

> So to prevent this, we could have created a different dataset dictionary from which the polars dataframe will be created, and using `None` to represent null instead of `np.nan` like this : 
``` python
data = {'animal': ['cat', 'cat', 'snake', 'dog', 'dog', 'cat', 'snake', 'cat', 'dog', 'dog'],
        'age': [2.5, 3, 0.5, None, 5, 2, 4.5, None, 7, 3],
        'visits': [1, 3, 2, 3, 2, 3, 1, 1, 2, 1],
        'priority': ['yes', 'yes', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no', 'no']}

labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']
```
> Or we could manually manipulate the column to replace nan with `None` like below

In [131]:
pldf = pldf.with_columns(
	pl.when(pl.col("age").is_nan())
	  .then(pl.lit(None).cast(pl.Float64))
	  .otherwise(pl.col("age"))
	  .alias("age")
)
pldf.describe()

statistic,index,animal,age,visits,priority
str,str,str,f64,f64,str
"""count""","""10""","""10""",8.0,10.0,"""10"""
"""null_count""","""0""","""0""",2.0,0.0,"""0"""
"""mean""",null,null,3.4375,1.9,null
"""std""",null,null,2.007797,0.875595,null
"""min""","""a""","""cat""",0.5,1.0,"""no"""
"""25%""",null,null,2.5,1.0,null
"""50%""",null,null,3.0,2.0,null
"""75%""",null,null,4.5,3.0,null
"""max""","""j""","""snake""",7.0,3.0,"""yes"""


## Puzzle 6.
 Return the first 3 rows of the DataFrame `df`.

In [132]:
# Return first 3 rows - pandas 
df.head(3)

,animal,age,visits,priority
a,cat,2.5,1,yes
b,cat,3.0,3,yes
c,snake,0.5,2,no


In [133]:
# Return first 3 rows - polars

pldf.head(3)

index,animal,age,visits,priority
str,str,f64,i64,str
"""a""","""cat""",2.5,1,"""yes"""
"""b""","""cat""",3.0,3,"""yes"""
"""c""","""snake""",0.5,2,"""no"""


## Puzzle  7.
 Select just the 'animal' and 'age' columns from the DataFrame `df`.

In [134]:
df[['animal','age']]

,animal,age
a,cat,2.5
b,cat,3.0
c,snake,0.5
d,dog,NaN
e,dog,5.0
f,cat,2.0
g,snake,4.5
h,cat,NaN
i,dog,7.0
j,dog,3.0


In [135]:
pldf.select(["animal", "age"])

animal,age
str,f64
"""cat""",2.5
"""cat""",3.0
"""snake""",0.5
"""dog""",null
"""dog""",5.0
"""cat""",2.0
"""snake""",4.5
"""cat""",null
"""dog""",7.0


> this is one of the key difference in how pandas works vs polars : how dataframe is sliced

> loc[] and iloc[] are not available in polars ( again : because no index). Because of this you need to rely on expressions like [`.select()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.select.html#polars.DataFrame.select) and .[`filter()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html#polars.DataFrame.filter)

## Puzzle  8
 Select the data in rows `[3, 4, 8]` *and* in columns `['animal', 'age']`.

In [136]:
df[['animal','age']][3:9]

# but this is preferred : df.loc['d':'i',['animal','age']] ... more on that later 

,animal,age
d,dog,NaN
e,dog,5.0
f,cat,2.0
g,snake,4.5
h,cat,NaN
i,dog,7.0


> PS: the code above works but this is referred to as **Chained Indexing** (ie chaining multiple squared brackets, or a aquared bracket with iloc[] or loc[] ) and is considered a bad technique when you use it in an assignment operation. So you will be better off not getting used to it.

> when you perform chained indexing in pandas, there is chance that you are pointing to  a temporary copy of the dataframe that was created as a result of that operation. So if you follow that with an assignment operation, there is a chance that you will be writing back to that temporary copy. When this happens, pandas gives you the  `SettingWithCopyWarning` warning ... but it will not stop your code, which could silenty introduce a 'false assignment' error to your pipeline 

> We will explore the right way to do this in a coming exercise

In [137]:
# Recommended polars solution 
pldf.select(['animal','age']).head(9).tail(6)

#Alternate solution (not recommended) : 
#
# ->   pldf[3:9]['animal','age'] or pldf[3:9,['animal','age']]
# we just sliced a polars dataframe using row index and this is why it is considered bad practice 
# read more in the article linked below
# alternatively you will find a short explanation about this in an upcoming puzzle 

animal,age
str,f64
"""dog""",null
"""dog""",5.0
"""cat""",2.0
"""snake""",4.5
"""cat""",null
"""dog""",7.0


> in polars  we will not have to worry about avoiding chained indexing's 'false assignment' errors because polars will outrightly stop your code when it sees a chanied [] followed by an assignment, with a `TypeError` error  

> square bracket slicing `pldf[]` is possible but not recommended, and  ***neither iloc[] or loc[] exist in polars*** 

> [read more !](https://kevinheavey.github.io/modern-polars/indexing.html#indexing)

> For this puzzle we are introducing 3  expressions 
> - .select() which is used to select columns just like SQL's SELECT
> - .head() equivalent to pandas head()
>- .tail() equivalent to pandas tail()

## Puzzle 9.
 Select only the rows where the number of visits is greater than 3.

In [138]:
# pandas 
df[df.visits > 3]

,animal,age,visits,priority


In [139]:
# polars 
pldf.filter(pl.col("visits") > 3)

index,animal,age,visits,priority
str,str,f64,i64,str


> For this puzzle we have introduced these in polars: 
> - .filter() method 
> - and the [`pl.col()`](https://docs.pola.rs/api/python/stable/reference/expressions/col.html) constructor 

## Puzzle 10.
 Select the rows where the age is missing, i.e. it is `NaN`.

In [140]:
# pandas
df[df.age.isnull()] # PS: isna() is an alias for isnull()

,animal,age,visits,priority
d,dog,NaN,3,yes
h,cat,NaN,1,yes


In [141]:
# polars
pldf.filter(pl.col("age").is_null())

index,animal,age,visits,priority
str,str,f64,i64,str
"""d""","""dog""",null,3,"""yes"""
"""h""","""cat""",null,1,"""yes"""


## Puzzle  11.
 Select the rows where the animal is a cat *and* the age is less than 3.

In [142]:
# pandas
df[(df.animal=='cat') & (df.age< 3)]
# remember that Pandas does not like multiple conditions without parentheses separating them

,animal,age,visits,priority
a,cat,2.5,1,yes
f,cat,2.0,3,no


In [143]:
# polars 
pldf.filter((pl.col("animal") == "cat") & (pl.col("age") < 3))

index,animal,age,visits,priority
str,str,f64,i64,str
"""a""","""cat""",2.5,1,"""yes"""
"""f""","""cat""",2.0,3,"""no"""


## Puzzle  12.
 Select the rows where the age is between 2 and 4 (inclusive).

In [144]:
#pandas
df[df["age"].between(2, 4)]

# alternate solution
#    df[(df.age >= 2) & (df.age <= 4)]

,animal,age,visits,priority
a,cat,2.5,1,yes
b,cat,3.0,3,yes
f,cat,2.0,3,no
j,dog,3.0,1,no


In [145]:
#polars
pldf.filter(pl.col("age").is_between(2, 4))


index,animal,age,visits,priority
str,str,f64,i64,str
"""a""","""cat""",2.5,1,"""yes"""
"""b""","""cat""",3.0,3,"""yes"""
"""f""","""cat""",2.0,3,"""no"""
"""j""","""dog""",3.0,1,"""no"""


>  the [`is_between()`](https://docs.pola.rs/api/python/dev/reference/expressions/api/polars.Expr.is_between.html) method  can be used directly on a polars.Series or as part of an polars.Expr within operations like filter() or with_columns() to check if value(s) fall within a specified lower and upper bound

## Puzzle 13. 
Change the age in row 'f' to 1.5.

In [146]:
# pandas
df.loc["f","age"] = 1.5

> this is the puzzle where you understand why you should avoid chained indexing. Assuming we had written this instead `df.loc["f"].loc["age"] = 1.5` or `df.loc['f']['age'] = 1.5` or `df[5:6]["age"] = 1.5` then we would have gotten a `SettingWithCopyWarning` warning, BUT our notebook run will not stop running, which could introduce silent error to our whole process

In [147]:
# polars
pldf = pldf.with_columns(
    pl.when(pl.col("index") == "f")
    .then(1.5)
    .otherwise(pl.col("age"))
    .alias("age")
)

pldf

# Alternate, not recommened approach
#  -->  pldf[5,"age"] = 1.5  # Did we just sliced a polars dataframe using a row index again ???

index,animal,age,visits,priority
str,str,f64,i64,str
"""a""","""cat""",2.5,1,"""yes"""
"""b""","""cat""",3.0,3,"""yes"""
"""c""","""snake""",0.5,2,"""no"""
"""d""","""dog""",null,3,"""yes"""
"""e""","""dog""",5.0,2,"""no"""
"""f""","""cat""",1.5,3,"""no"""
"""g""","""snake""",4.5,1,"""no"""
"""h""","""cat""",null,1,"""yes"""
"""i""","""dog""",7.0,2,"""no"""


> You might be wondering why the long method was the preferred approach when we have a shorter alternative. 

> First off, polars dataframes are immutable, this means we are 'not supposed to be able to' change them in place using an assignment operation like in pandas.

> Because of this, the 'only' way to change a cell in both a polars dataframe and lazyframe  will be to use the with_columns() method to create a new column with the updated cell

> But looking at the alternative solution above, apparently you could still change a polars dataframe in place using squared brackets. But this 'Square Bracket Assignment' feature was addad as a "convenience feature" for interactive work and ***it will not work on a lazyframe*** . So in short you should not get used to it, just know that it exists for your convenience in certain situations


## Puzzle  14.
Calculate the sum of all visits in `df` (i.e. find the total number of visits).

In [148]:
# pandas
df["visits"].sum()

np.int64(19)

In [149]:
# polars
pldf.select(pl.col("visits").sum()).item()

19

> The polars dataframe's [`.item()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.item.html) method can be used to return the element at a given cell  or a 1x1 dataframe (which is the case above) as a scalar value

## Puzzle  15.
 Calculate the mean age for each different animal in `df`.

In [150]:
# pandas
df.groupby('animal')['age'].mean()

animal
cat      2.333333
dog      5.000000
snake    2.500000
Name: age, dtype: float64

In [151]:
#polars
print(pldf.group_by("animal", maintain_order=True).agg(pl.col("age").mean()))

shape: (3, 2)
┌────────┬──────────┐
│ animal ┆ age      │
│ ---    ┆ ---      │
│ str    ┆ f64      │
╞════════╪══════════╡
│ cat    ┆ 2.333333 │
│ snake  ┆ 2.5      │
│ dog    ┆ 5.0      │
└────────┴──────────┘


> The [`.group_by()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by.html) (Context): Segregates the data into "buckets" based on unique values in a column. after which you can then apply aggregation functions using [`.agg()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.agg.html#polars.dataframe.group_by.GroupBy.agg)

> The maintain_order=True parameter ensures that the group order is retained, but makes the process a little slower 

## Puzzle 16. 
Append a new row 'k' to `df` with your choice of values for each column. Then delete that row to return the original DataFrame.

In [152]:
# pandas 
df.loc['k'] = ['dog', 5.5, 2, 'no']
print(df)
df.drop('k', inplace=True)

  animal  age  visits priority
a    cat  2.5       1      yes
b    cat  3.0       3      yes
c  snake  0.5       2       no
d    dog  NaN       3      yes
e    dog  5.0       2       no
f    cat  1.5       3       no
g  snake  4.5       1       no
h    cat  NaN       1      yes
i    dog  7.0       2       no
j    dog  3.0       1       no
k    dog  5.5       2       no


In [153]:
# Polars
new_row = pl.DataFrame({"index": ["k"], "animal": ["dog"], "age": [5.5], "visits": [2], "priority": ["no"]})
# add the row
pldf = pl.concat([pldf, new_row])
pldf

# delete the row 
pldf = pldf.filter(pl.col("index") != "k")

> [`pl.concat()`](https://docs.pola.rs/api/python/stable/reference/api/polars.concat.html#polars.concat) (Function): Stitches two separate DataFrames or LazyFrames together vertically or horizontally.

> The mental logic for "dropping" inpolars is to run a "select" that leaves out the rows you intend to drop

## Puzzle 17.
 Count the number of each type of animal in `df`.

In [154]:
# pandas
df['animal'].value_counts()

animal
cat      4
dog      4
snake    2
Name: count, dtype: int64

In [155]:
# Polars
pldf.group_by("animal").len()

# Alternative solution 
#   --   pldf['animal'].value_counts()

animal,len
str,u32
"""dog""",4
"""snake""",2
"""cat""",4


> [`.len()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.len.html#polars.dataframe.group_by.GroupBy.len) (method): A group_by method that returns the number of rows in each group.

> [`.value_counts()`](https://docs.pola.rs/api/python/stable/reference/series/api/polars.Series.value_counts.html) (Method): A series method that Count the occurrences of unique values.

## Puzzle  18.
 Sort `df` first by the values in the 'age' in *decending* order, then by the value in the 'visits' column in *ascending* order (so row `i` should be first, and row `d` should be last).

In [156]:
# pandas
df.sort_values(by=['age', 'visits'], ascending=[False, True])

,animal,age,visits,priority
i,dog,7.0,2,no
e,dog,5.0,2,no
g,snake,4.5,1,no
j,dog,3.0,1,no
b,cat,3.0,3,yes
a,cat,2.5,1,yes
f,cat,1.5,3,no
c,snake,0.5,2,no
h,cat,NaN,1,yes
d,dog,NaN,3,yes


In [157]:
# polars
pldf.sort(["age", "visits"], descending=[True, False])

index,animal,age,visits,priority
str,str,f64,i64,str
"""h""","""cat""",null,1,"""yes"""
"""d""","""dog""",null,3,"""yes"""
"""i""","""dog""",7.0,2,"""no"""
"""e""","""dog""",5.0,2,"""no"""
"""g""","""snake""",4.5,1,"""no"""
"""j""","""dog""",3.0,1,"""no"""
"""b""","""cat""",3.0,3,"""yes"""
"""a""","""cat""",2.5,1,"""yes"""
"""f""","""cat""",1.5,3,"""no"""


> Quick mental note: Pandas' sort_values() uses the ascending= parameter while polars' sort() used descending=

> We have introduced the polars [.sort()](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.sort.html) dataframe method here 

## Puzzle 19.
The 'priority' column contains the values 'yes' and 'no'. Replace this column with a column of boolean values: 'yes' should be `True` and 'no' should be `False`.

In [158]:
df['priority'] = df['priority'].map({'yes': True, 'no': False})
df 

# Alternate approach
# -- df = df.assign(priority= df['priority'].map({'yes': True, 'no': False}))

,animal,age,visits,priority
a,cat,2.5,1,True
b,cat,3.0,3,True
c,snake,0.5,2,False
d,dog,NaN,3,True
e,dog,5.0,2,False
f,cat,1.5,3,False
g,snake,4.5,1,False
h,cat,NaN,1,True
i,dog,7.0,2,False
j,dog,3.0,1,False


In [159]:
# polars 
pldf.with_columns(pl.col("priority") == "yes")

index,animal,age,visits,priority
str,str,f64,i64,bool
"""a""","""cat""",2.5,1,true
"""b""","""cat""",3.0,3,true
"""c""","""snake""",0.5,2,false
"""d""","""dog""",null,3,true
"""e""","""dog""",5.0,2,false
"""f""","""cat""",1.5,3,false
"""g""","""snake""",4.5,1,false
"""h""","""cat""",null,1,true
"""i""","""dog""",7.0,2,false


## Puzzle 20 
In the 'animal' column, change the 'snake' entries to 'python'.

In [160]:
df["animal"] = df['animal'].replace("snake", "python")
df

,animal,age,visits,priority
a,cat,2.5,1,True
b,cat,3.0,3,True
c,python,0.5,2,False
d,dog,NaN,3,True
e,dog,5.0,2,False
f,cat,1.5,3,False
g,python,4.5,1,False
h,cat,NaN,1,True
i,dog,7.0,2,False
j,dog,3.0,1,False


Knowing when to use map() or replace() in pandas : 
>You cannot use map to change values in a column when you don't know all the possible values that already exist in the column. If you do so , any value that you miss in your map will be replaced with Nan. For cases like this , you can just use replace() to change only the values that you know

In [161]:
# polars
pldf = pldf.with_columns(pl.col("animal").replace("snake", "python"))
pldf

index,animal,age,visits,priority
str,str,f64,i64,str
"""a""","""cat""",2.5,1,"""yes"""
"""b""","""cat""",3.0,3,"""yes"""
"""c""","""python""",0.5,2,"""no"""
"""d""","""dog""",null,3,"""yes"""
"""e""","""dog""",5.0,2,"""no"""
"""f""","""cat""",1.5,3,"""no"""
"""g""","""python""",4.5,1,"""no"""
"""h""","""cat""",null,1,"""yes"""
"""i""","""dog""",7.0,2,"""no"""


> [`replace()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.replace.html#polars.Expr.replace) : an expression method, replaces a single value by another value. Values that were not replaced remain unchanged.

## Puzzle 21.** For each animal type and each number of visits, find the mean age. In other words, each row is an animal, each column is a number of visits and the values are the mean ages (*hint: use a pivot table*).

In [162]:
# pandas
df.pivot_table(index='animal', columns='visits', values='age', aggfunc='mean')

visits,1,2,3
animal,,,
cat,2.5,NaN,2.25
dog,3.0,6.0,NaN
python,4.5,0.5,NaN


In [163]:
# polars 
pldf.pivot(values="age", index="animal", on="visits", aggregate_function="mean")

animal,1,3,2
str,f64,f64,f64
"""cat""",2.5,2.25,null
"""python""",4.5,null,0.5
"""dog""",3.0,null,6.0


[`.pivot()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html#polars.DataFrame.pivot) (datafrane Method): A complex reshape operation that turns unique values of one column into new column headers.

Bonus Method that i haven't but is inportnt for any polars fundamentals material:
> [`.alias()`](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.alias.html#polars.Expr.alias) : an expression method that Renames the expression.

# Puzzle  22 - 28

## DataFrames: beyond the basics

### Slightly trickier: you may need to combine two or more methods to get the right answer

Difficulty: *medium*

The previous section was tour through some basic but essential DataFrame operations. Below are some ways that you might need to cut your data, but for which there is no single "out of the box" method.

## Puzzle 22 
You have a DataFrame `df` with a column 'A' of integers. For example:
```python
df = pd.DataFrame({'A': [1, 2, 2, 3, 4, 5, 5, 5, 6, 7, 7]})
```

How do you filter out rows which contain the same integer as the row immediately above?

You should be left with a column containing the following values:

```python
1, 2, 3, 4, 5, 6, 7
```

In [164]:
# pandas solution -- this is the 'deduplicating consecutive values' problem in data analytics , 
# which is different from deduplicating the whole column which your first instinct might tell you to
df = pd.DataFrame({'A': [1, 2, 2, 3, 4, 5, 5, 5, 6, 7, 7]})
result = df.loc[df['A'].shift() != df['A'] ]
result

,A
0,1
1,2
3,3
4,4
5,5
8,6
9,7


In [165]:
# polars solution
pldf = pl.DataFrame({'A': [1, 2, 2, 3, 4, 5, 5, 5, 6, 7, 7]})
result = pldf.filter(
    (pl.col("A") != pl.col("A").shift()).fill_null(True)
)
result

A
i64
1
2
3
4
5
6
7


## Puzzle 23. 
Given a DataFrame of numeric values, say
```python
df = pd.DataFrame(np.random.random(size=(5, 3))) # a 5x3 frame of float values
```

how do you subtract the row mean from each element in the row?

In [166]:
# pandas 
df = pd.DataFrame(np.random.random(size=(5, 3)))

row_means = df.mean(axis=1)

# 2. Subtract the means, aligning them with the rows (axis=0)
df.sub(row_means, axis=0)

,0,1,2
0,0.015045,-0.285128,0.270083
1,0.117396,-0.283677,0.166280
2,0.051294,-0.351117,0.299823
3,-0.007329,0.210216,-0.202888
4,-0.001937,-0.232436,0.234374


In [167]:
#polars 

# to make the outut consistent we will make sure 
# the polars dataframe is a copy of the pandas one 

pldf = pl.from_pandas(df)

# Use horizontal mean across all columns
pldf.with_columns(pl.all() - pl.mean_horizontal(pl.all()))

0,1,2
f64,f64,f64
0.015045,-0.285128,0.270083
0.117396,-0.283677,0.16628
0.051294,-0.351117,0.299823
-0.007329,0.210216,-0.202888
-0.001937,-0.232436,0.234374


## Puzzle 24 
Suppose you have DataFrame with 10 columns of real numbers, for example:

```python
df = pd.DataFrame(np.random.random(size=(5, 10)), columns=list('abcdefghij'))
```
Which column of numbers has the smallest sum?  Return that column's label.

In [168]:
# ---- pandas
df = pd.DataFrame(np.random.random(size=(5, 10)), columns=list('abcdefghij'))

# 1. Sum the columns (returns a Series)
# 2. Get the label of the minimum value
min_col_label = df.sum().idxmin()

print(f"The column with the smallest sum is: {min_col_label}")

The column with the smallest sum is: h


In [169]:
# ----- polars 

pldf = pl.DataFrame(np.random.random(size=(5, 10)), schema=list('abcdefghij'))

# 1. Sum all columns (returns a 1-row DataFrame)
# 2. Melt to turn column names into a 'variable' column
# 3. Sort by the sums and take the first column name
min_col_label = (
    pldf.sum()
    .unpivot()
    .sort("value")
    .item(0, "variable")
)

print(f"The column with the smallest sum is: {min_col_label}")

The column with the smallest sum is: d


## Puzzle 25. 
How do you count how many unique rows a DataFrame has (i.e. ignore all rows that are duplicates)? As input, use a DataFrame of zeros and ones with 10 rows and 3 columns.

```python
df = pd.DataFrame(np.random.randint(0, 2, size=(10, 3)))
```

In [170]:
# ------ pandas 
df = pd.DataFrame(np.random.randint(0, 2, size=(10, 3)))

# 1. Drop duplicate rows
# 2. Get the number of rows remaining
unique_row_count = len(df.drop_duplicates())

print(f"Number of unique rows: {unique_row_count}")

Number of unique rows: 7


In [171]:
# ------ polars
pldf = pl.from_pandas(df)

# 1. Keep only unique rows
# 2. Get the height (number of rows)
unique_row_count = pldf.unique().height

print(f"Number of unique rows: {unique_row_count}")

Number of unique rows: 7


## Puzzle 26 - 28

The next three puzzles are slightly harder.


**26.** In the cell below, you have a DataFrame `df` that consists of 10 columns of floating-point numbers. Exactly 5 entries in each row are NaN values. 

For each row of the DataFrame, find the *column* which contains the *third* NaN value.

You should return a Series of column labels: `e, c, d, h, d`

In [172]:
nan = np.nan

data = [[0.04,  nan,  nan, 0.25,  nan, 0.43, 0.71, 0.51,  nan,  nan],
        [ nan,  nan,  nan, 0.04, 0.76,  nan,  nan, 0.67, 0.76, 0.16],
        [ nan,  nan, 0.5 ,  nan, 0.31, 0.4 ,  nan,  nan, 0.24, 0.01],
        [0.49,  nan,  nan, 0.62, 0.73, 0.26, 0.85,  nan,  nan,  nan],
        [ nan,  nan, 0.41,  nan, 0.05,  nan, 0.61,  nan, 0.48, 0.68]]

columns = list('abcdefghij')

df = pd.DataFrame(data, columns=columns)

# ------ pandas 


# 1. Create a boolean mask of NaNs
# 2. Cumulative sum across columns (axis=1). True counts as 1.
# 3. Find where the cumulative sum is exactly 3
# 4. idxmax along axis 1 returns the first column label where the condition is True
result = (df.isna().cumsum(axis=1) == 3).idxmax(axis=1)

print(result.tolist())

['e', 'c', 'd', 'h', 'd']


In [173]:
# ------ polars

pldf = pl.from_pandas(df)
# 1. Map column names to their index
column_names = pldf.columns

# 2. Use a horizontal expression to find column indices where values are null
# 3. Extract the 3rd index (index 2) from that list
result = pldf.select(
    pl.struct(pl.all())
    .map_elements(lambda row: [k for k, v in row.items() if v is None][2], return_dtype=pl.String)
).to_series()

print(result.to_list())

['e', 'c', 'd', 'h', 'd']


## Puzzle 27. A DataFrame has a column of groups 'grps' and and column of integer values 'vals': 

```python
df = pd.DataFrame({'grps': list('aaabbcaabcccbbc'), 
                   'vals': [12,345,3,1,45,14,4,52,54,23,235,21,57,3,87]})
```
For each *group*, find the sum of the three greatest values. You should end up with the answer as follows:
```
grps
a    409
b    156
c    345
```

In [174]:
# ------ pandas

df = pd.DataFrame({
    'grps': list('aaabbcaabcccbbc'), 
    'vals': [12,345,3,1,45,14,4,52,54,23,235,21,57,3,87]
})

# 1. Group by 'grps'
# 2. Select 'vals' column
# 3. Use nlargest(3) to get top values per group
# 4. Sum the results per group
result = df.groupby('grps')['vals'].nlargest(3).groupby(level=0).sum()

print(result)

grps
a    409
b    156
c    345
Name: vals, dtype: int64


In [175]:
# ------ polars
pldf = pl.DataFrame({
    'grps': list('aaabbcaabcccbbc'), 
    'vals': [12,345,3,1,45,14,4,52,54,23,235,21,57,3,87]
})

# 1. Group by 'grps'
# 2. Inside each group: sort 'vals' descending, take the first 3, and sum them
result = (
    pldf.group_by("grps")
    .agg(
        pl.col("vals").sort(descending=True).head(3).sum()
    )
    .sort("grps")
)

print(result)

shape: (3, 2)
┌──────┬──────┐
│ grps ┆ vals │
│ ---  ┆ ---  │
│ str  ┆ i64  │
╞══════╪══════╡
│ a    ┆ 409  │
│ b    ┆ 156  │
│ c    ┆ 345  │
└──────┴──────┘


## Puzzle 28. 
The DataFrame `df` constructed below has two integer columns 'A' and 'B'. The values in 'A' are between 1 and 100 (inclusive). 

For each group of 10 consecutive integers in 'A' (i.e. `(0, 10]`, `(10, 20]`, ...), calculate the sum of the corresponding values in column 'B'.

The answer should be a Series as follows:

```
A
(0, 10]      635
(10, 20]     360
(20, 30]     315
(30, 40]     306
(40, 50]     750
(50, 60]     284
(60, 70]     424
(70, 80]     526
(80, 90]     835
(90, 100]    852
```

In [176]:
df = pd.DataFrame(np.random.RandomState(8765).randint(1, 101, size=(100, 2)), columns = ["A", "B"])

# ------ pandas

# 1. Define the bin edges (0, 10, 20, ..., 100)
bins = np.arange(0, 101, 10)

# 2. Use pd.cut to assign each 'A' value to a bin
# 3. Group by these bins and sum 'B'
result = df.groupby(pd.cut(df['A'], bins))['B'].sum()

print(result)

A
(0, 10]      635
(10, 20]     360
(20, 30]     315
(30, 40]     306
(40, 50]     750
(50, 60]     284
(60, 70]     424
(70, 80]     526
(80, 90]     835
(90, 100]    852
Name: B, dtype: int32


C:\Users\USER\AppData\Local\Temp\ipykernel_5104\3073247385.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = df.groupby(pd.cut(df['A'], bins))['B'].sum()


In [177]:
pldf = pl.from_pandas(df)

# 1. Create a grouping column using math: ((A-1) // 10) gives 0 for 1-10, 1 for 11-20, etc.
# 2. Map those integers back to the string labels for the final output
result = (
    pldf.with_columns(
        group_id=((pl.col("A") - 1) // 10)
    )
    .group_by("group_id")
    .agg(pl.col("B").sum())
    .sort("group_id")
    .with_columns(
        A=(pl.col("group_id") * 10).cast(pl.String) + "-" + (pl.col("group_id") * 10 + 10).cast(pl.String)
    )
    .select(["A", "B"])
)

print(result)

shape: (10, 2)
┌────────┬─────┐
│ A      ┆ B   │
│ ---    ┆ --- │
│ str    ┆ i32 │
╞════════╪═════╡
│ 0-10   ┆ 635 │
│ 10-20  ┆ 360 │
│ 20-30  ┆ 315 │
│ 30-40  ┆ 306 │
│ 40-50  ┆ 750 │
│ 50-60  ┆ 284 │
│ 60-70  ┆ 424 │
│ 70-80  ┆ 526 │
│ 80-90  ┆ 835 │
│ 90-100 ┆ 852 │
└────────┴─────┘


## Puzzle 29 - 32

## DataFrames: harder problems 

### These might require a bit of thinking outside the box...

...but all are solvable using just the usual pandas/NumPy methods (and so avoid using explicit `for` loops).

Difficulty: *hard*

## Puzzle 29 
Consider a DataFrame `df` where there is an integer column 'X':
```python
df = pd.DataFrame({'X': [7, 2, 0, 3, 4, 2, 5, 0, 3, 4]})
```
For each value, count the difference back to the previous zero (or the start of the Series, whichever is closer). These values should therefore be 

```
[1, 2, 0, 1, 2, 3, 4, 0, 1, 2]
```

Make this a new column 'Y'.

In [178]:
df = pd.DataFrame({'X': [7, 2, 0, 3, 4, 2, 5, 0, 3, 4]})

# 1. Identify where X is 0
izero = (df['X'] == 0)

# 2. Create a 'group' ID that increments every time a 0 appears
# This gives us: [0, 0, 1, 1, 1, 1, 1, 2, 2, 2]
abs_zeros = izero.cumsum()

# 3. Use cumulative count within these groups.
# We then add 1 to the count, but reset it to 0 specifically where the original value was 0.
df['Y'] = df.groupby(abs_zeros).cumcount()

# The first group (before any zero) needs an offset because there was no zero to 'start' it
# We find the index of the first zero to determine how to adjust the first group
first_zero_idx = (df['X'] == 0).idxmax()
df.loc[:first_zero_idx-1, 'Y'] += 1
df.loc[df['X'] == 0, 'Y'] = 0

print(df['Y'].tolist())

[1, 2, 0, 1, 2, 3, 4, 0, 1, 2]


In [179]:
# ------ polars

pldf = pl.from_pandas(df)

# 1. Create a boolean column for zeros
# 2. Use cum_sum on the booleans to create "windows" or "groups"
# 3. Inside those windows, count the rows
result = pldf.with_columns(
    group=pl.col("X").eq(0).cum_sum()
).with_columns(
    Y=pl.int_range(0, pl.len()).over("group")
)

# Adjusting for the 'pre-zero' values (they should count from the start)
# and ensuring zeros stay zero.
first_zero_index = pldf.select(pl.arg_where(pl.col("X") == 0)).item(0, 0)

result = result.with_columns(
    Y = pl.when(pl.col("X") == 0)
        .then(0)
        .when(pl.col("group") == 0)
        .then(pl.col("Y") + 1)
        .otherwise(pl.col("Y"))
).select("X", "Y")

print(result["Y"].to_list())

[1, 2, 0, 1, 2, 3, 4, 0, 1, 2]


**30.** Consider the DataFrame constructed below which contains rows and columns of numerical data. 

Create a list of the column-row index locations of the 3 largest values in this DataFrame. In this case, the answer should be:
```
[(5, 7), (6, 4), (2, 5)]
```

In [180]:
df = pd.DataFrame(np.random.RandomState(30).randint(1, 101, size=(8, 8)))

# 1. Unstack the dataframe into a Series (this creates a MultiIndex of [col, row])
# 2. Sort by values descending
# 3. Take the top 3 and extract the index (the coordinates)
df.unstack().sort_values(ascending=False).head(3).index.tolist()

[(2, 5), (5, 7), (6, 4)]

In [181]:
# ------ polars 
pldf = pl.from_pandas(df)

# 1. Add a row index so we can track coordinates
# 2. Melt to long format (columns: row_idx, variable (column_name), value)
# 3. Sort by value and take top 3
# 4. Extract (column, row) pairs
top_3 = (
    pldf.with_row_index("row_idx")
    .unpivot(index="row_idx")
    .sort("value", descending=True)
    .head(3)
)

# Convert to the specific list of tuples format (column_int, row_int)
result = [
    (int(col), row) 
    for col, row in zip(top_3["variable"], top_3["row_idx"])
]

print(result)


[(2, 5), (5, 7), (6, 4)]


**31.** You are given the DataFrame below with a column of group IDs, 'grps', and a column of corresponding integer values, 'vals'.

```python
df = pd.DataFrame({"vals": np.random.RandomState(31).randint(-30, 30, size=15), 
                   "grps": np.random.RandomState(31).choice(["A", "B"], 15)})
```

Create a new column 'patched_values' which contains the same values as the 'vals' any negative values in 'vals' with the group mean:

```
    vals grps  patched_vals
0    -12    A          13.6
1     -7    B          28.0
2    -14    A          13.6
3      4    A           4.0
4     -7    A          13.6
5     28    B          28.0
6     -2    A          13.6
7     -1    A          13.6
8      8    A           8.0
9     -2    B          28.0
10    28    A          28.0
11    12    A          12.0
12    16    A          16.0
13   -24    A          13.6
14   -12    A          13.6
```

In [182]:
df = pd.DataFrame({"vals": np.random.RandomState(31).randint(-30, 30, size=15), 
                   "grps": np.random.RandomState(31).choice(["A", "B"], 15)})

# 1. Calculate the mean of POSITIVE values only for each group
# We mask negative values to NaN so they don't influence the mean
def mean_positive(x):
    return x[x > 0].mean()

group_means = df.groupby('grps')['vals'].transform(mean_positive)

# 2. Use np.where to choose between the original value and the group mean
df['patched_vals'] = np.where(df['vals'] < 0, group_means, df['vals'])

df

,vals,grps,patched_vals
0,-12,A,13.6
1,-7,B,28.0
2,-14,A,13.6
3,4,A,4.0
4,-7,A,13.6
5,28,B,28.0
6,-2,A,13.6
7,-1,A,13.6
8,8,A,8.0
9,-2,B,28.0


In [183]:
# ------ polars

pldf = pl.from_pandas(df)

# 1. Calculate the mean of values > 0 per group using a window expression
# 2. Use when/then logic to apply the patch
result = pldf.with_columns(
    patched_vals = pl.when(pl.col("vals") < 0)
    .then(
        pl.col("vals").filter(pl.col("vals") > 0).mean().over("grps")
    )
    .otherwise(pl.col("vals"))
)

result

vals,grps,patched_vals
i32,str,f64
-12,"""A""",13.6
-7,"""B""",28.0
-14,"""A""",13.6
4,"""A""",4.0
-7,"""A""",13.6
…,…,…
28,"""A""",28.0
12,"""A""",12.0
16,"""A""",16.0


**32.** Implement a rolling mean over groups with window size 3, which ignores NaN value. For example consider the following DataFrame:

```python
>>> df = pd.DataFrame({'group': list('aabbabbbabab'),
                       'value': [1, 2, 3, np.nan, 2, 3, np.nan, 1, 7, 3, np.nan, 8]})
>>> df
   group  value
0      a    1.0
1      a    2.0
2      b    3.0
3      b    NaN
4      a    2.0
5      b    3.0
6      b    NaN
7      b    1.0
8      a    7.0
9      b    3.0
10     a    NaN
11     b    8.0
```
The goal is to compute the Series:

```
0     1.000000
1     1.500000
2     3.000000
3     3.000000
4     1.666667
5     3.000000
6     3.000000
7     2.000000
8     3.666667
9     2.000000
10    4.500000
11    4.000000
```
E.g. the first window of size three for group 'b' has values 3.0, NaN and 3.0 and occurs at row index 5. Instead of being NaN the value in the new column at this row index should be 3.0 (just the two non-NaN values are used to compute the mean (3+3)/2)

In [184]:
df = pd.DataFrame({'group': list('aabbabbbabab'),
                   'value': [1, 2, 3, np.nan, 2, 3, np.nan, 1, 7, 3, np.nan, 8]})

# 1. Group by 'group'
# 2. Select the 'value' column
# 3. Apply a rolling window of 3
# 4. min_periods=1 allows the window to calculate even if 1 or 2 values are NaN
g1 = df.groupby(['group'])['value']
result = g1.rolling(3, min_periods=1).mean()

# The rolling operation on GroupBy returns a MultiIndex (group, original_index)
# We sort by the second level of the index to return to original order
result = result.reset_index(level=0, drop=True).sort_index()

print(result)

0     1.000000
1     1.500000
2     3.000000
3     3.000000
4     1.666667
5     3.000000
6     3.000000
7     2.000000
8     3.666667
9     2.000000
10    4.500000
11    4.000000
Name: value, dtype: float64


In [185]:
# ------ polars 

pldf = pl.DataFrame({'group': list('aabbabbbabab'),
                   'value': [1, 2, 3, None, 2, 3, None, 1, 7, 3, None, 8]})

# 1. We use the rolling_mean expression
# 2. window_size=3
# 3. min_periods=1 (same logic as pandas)
# 4. .over("group") ensures the window doesn't leak across different groups
result = pldf.select(
    pl.col("value").rolling_mean(window_size=3, min_samples=1).over("group")
)

result.to_series()

value
f64
1.0
1.5
3.0
3.0
1.666667
…
2.0
3.666667
2.0


## Series and DatetimeIndex

### Exercises for creating and manipulating Series with datetime data

Difficulty: *easy/medium*

pandas is fantastic for working with dates and times. These puzzles explore some of this functionality.

**33.** Create a DatetimeIndex that contains each business day of 2015 and use it to index a Series of random numbers. Let's call this Series `s`.

In [186]:
# pandas 
# 1. Create the DatetimeIndex for business days in 2015
dti = pd.date_range(start='2015-01-01', end='2015-12-31', freq='B')

# 2. Create the Series with random numbers indexed by those dates
s = pd.Series(np.random.rand(len(dti)), index=dti)

print(s.head())
print(f"\nTotal business days in 2015: {len(s)}")

2015-01-01    0.483753
2015-01-02    0.235357
2015-01-05    0.362058
2015-01-06    0.804475
2015-01-07    0.323819
Freq: B, dtype: float64

Total business days in 2015: 261


In [187]:
# polars 

from datetime import date

# 1. Generate every day in 2015
# 2. Filter for business days (Monday=1, ..., Friday=5)
# 3. Add a column of random values
pldf = (
    pl.date_range(date(2015, 1, 1), date(2015, 12, 31), interval="1d", eager=True)
    .alias("date")
    .to_frame()
    .filter(pl.col("date").dt.weekday() < 6) # 6 and 7 are Saturday/Sunday
    .with_columns(
        vals = pl.lit(np.random.rand(261)) # 261 is the count of B-days in 2015
    )
)

print(pldf.head())

shape: (5, 2)
┌────────────┬──────────┐
│ date       ┆ vals     │
│ ---        ┆ ---      │
│ date       ┆ f64      │
╞════════════╪══════════╡
│ 2015-01-01 ┆ 0.96902  │
│ 2015-01-02 ┆ 0.565656 │
│ 2015-01-05 ┆ 0.523831 │
│ 2015-01-06 ┆ 0.701921 │
│ 2015-01-07 ┆ 0.098463 │
└────────────┴──────────┘


> Indexing Differences: Again Pandas attaches dates as a metadata Index , whereas Polars treats dates as a standard data column since it lacks a built-in index.

> Function Similarity: Both libraries provide a **date_range** function to generate sequences of time points.

> Calendar Awareness: Pandas is "calendar-aware" out of the box; it uses [offset aliases](https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-offset-aliases) like freq='B' which handles business day logic automatically.

> Explicit Logic: Polars requires an explicit .filter() step (e.g., checking weekday) to achieve the same result as a Pandas offset alias.


**34.** Find the sum of the values in series `s` for every Wednesday.

In [188]:
# pandas 

# 1. Access the weekday property of the index (Monday=0, Wednesday=2)
# 2. Filter and sum
wednesday_sum = s[s.index.weekday == 2].sum()

print(f"Sum of values on Wednesdays: {wednesday_sum}")

Sum of values on Wednesdays: 24.264280941959296


In [189]:
# Polars 
# 1. Filter the 'date' column where the weekday is 3 (Polars uses 1-7, Mon-Sun)
# 2. Select the 'vals' column and sum
wednesday_sum = pldf.filter(pl.col("date").dt.weekday() == 3).select(pl.col("vals").sum()).item()

print(f"Sum of values on Wednesdays: {wednesday_sum}")

Sum of values on Wednesdays: 30.56221745110558


> Direct Access: Pandas allows direct attribute access through the index (e.g., s.index.weekday), making it very concise for Series.

> The Namespace: Polars requires the .dt namespace to access datetime-specific functions like weekday() on a column.

> Integer Mapping: Note the numbering difference: Pandas defaults to Monday=0, while Polars defaults to Monday=1.

> Scalar Extraction: To get a single number out of a Polars result, we use .item(), whereas Pandas returns a scalar directly from the sum.

**35.** For each calendar month in `s`, find the mean of values.

In [190]:
# ------ Pandas
s.resample('ME').mean()

2015-01-31    0.513277
2015-02-28    0.477536
2015-03-31    0.562327
2015-04-30    0.513159
2015-05-31    0.483536
2015-06-30    0.513872
2015-07-31    0.490192
2015-08-31    0.637844
2015-09-30    0.449590
2015-10-31    0.458470
2015-11-30    0.506259
2015-12-31    0.340393
Freq: ME, dtype: float64

In [191]:
# --- polars 

# 1. group_by_dynamic uses the 'date' column
# 2. '1mo' tells Polars to create 1-month windows
monthly_means = pldf.group_by_dynamic("date", every="1mo").agg(
    pl.col("vals").mean()
)

print(monthly_means)

shape: (12, 2)
┌────────────┬──────────┐
│ date       ┆ vals     │
│ ---        ┆ ---      │
│ date       ┆ f64      │
╞════════════╪══════════╡
│ 2015-01-01 ┆ 0.48908  │
│ 2015-02-01 ┆ 0.515044 │
│ 2015-03-01 ┆ 0.538075 │
│ 2015-04-01 ┆ 0.428577 │
│ 2015-05-01 ┆ 0.541714 │
│ …          ┆ …        │
│ 2015-08-01 ┆ 0.561473 │
│ 2015-09-01 ┆ 0.550468 │
│ 2015-10-01 ┆ 0.516235 │
│ 2015-11-01 ┆ 0.500977 │
│ 2015-12-01 ┆ 0.448936 │
└────────────┴──────────┘


* You can also use the resample() on a dataframe if it has a date-time like ([DatetimeIndex](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.html), [TimedeltaIndex](https://pandas.pydata.org/docs/reference/api/pandas.TimedeltaIndex.html) or [PeriodIndex](https://pandas.pydata.org/docs/reference/api/pandas.PeriodIndex.html)) index or column 

* Running the resample() method on a valid (ie contains a date-time like index ) object will only return an DatetimeIndexResampler object, to get some actual data back , you need to use an agregat method e.g `sum()`, `mean()` on it.. you will notice it behaved just like using the `groupby()` method well, it is "the groupby for timeseries data"

* the group_by_dynamic() method is polars equivalent of pandas .resample() .. its too is used to group a polars df based on a time column (or index value of type Int32, Int64)
    * it returns a DynamicGroupBym unless you run a .agg expression on it 

**36.** For each group of four consecutive calendar months in `s`, find the date on which the highest value occurred.

In [201]:
# ------ Pandas
s.groupby(pd.Grouper(freq='4ME')).idxmax()

2015-01-31   2015-01-09
2015-05-31   2015-02-11
2015-09-30   2015-08-17
2016-01-31   2015-11-03
Freq: 4ME, dtype: datetime64[ns]

In [199]:
# ------ Polars 
# 1. Use group_by_dynamic with a 4-month window
# 2. Find the row (date) where 'vals' is at its maximum within that 4-month block
#result = pldf.group_by_dynamic("date", every="1mo", period="4mo").agg(
#    pl.col("date").sort_by("vals", descending=True).first()
#)

result = pldf.group_by_dynamic("date", every="1mo", period="4mo").agg(
    pl.col("date").sort_by("vals", descending=True).first().alias("max_val_date")
)

print(result.tail())

shape: (5, 2)
┌────────────┬──────────────┐
│ date       ┆ max_val_date │
│ ---        ┆ ---          │
│ date       ┆ date         │
╞════════════╪══════════════╡
│ 2015-08-01 ┆ 2015-10-23   │
│ 2015-09-01 ┆ 2015-10-23   │
│ 2015-10-01 ┆ 2015-10-23   │
│ 2015-11-01 ┆ 2015-11-25   │
│ 2015-12-01 ┆ 2015-12-23   │
└────────────┴──────────────┘


**37.** Create a DateTimeIndex consisting of the third Thursday in each month for the years 2015 and 2016.

In [202]:
# ------ Pandas
# 'WOM-3THU' = Week Of Month, 3rd Thursday
thursdays = pd.date_range('2015-01-01', '2015-12-31', freq='WOM-3THU')

print(thursdays)

DatetimeIndex(['2015-01-15', '2015-02-19', '2015-03-19', '2015-04-16',
               '2015-05-21', '2015-06-18', '2015-07-16', '2015-08-20',
               '2015-09-17', '2015-10-15', '2015-11-19', '2015-12-17'],
              dtype='datetime64[ns]', freq='WOM-3THU')


In [207]:
# ------polars
# thursdays = (
#     pl.date_range(date(2015, 1, 1), date(2015, 12, 31), interval="1d", eager=True)
#     .to_frame("date")
#     .filter(pl.col("date").dt.weekday() == 4) # Thursday
#     .group_by(pl.col("date").dt.month())
#     .agg(pl.col("date").sort().gather(2)) # Index 2 is the 3rd occurrence
#     .sort("date")
#     .explode("date")
#     .get_column("date")
# )

thursdays = (
    pl.date_range(date(2015, 1, 1), date(2015, 12, 31), interval="1d", eager=True)
    .to_frame("date")
    .filter(pl.col("date").dt.weekday() == 4) # Thursday
    .group_by(pl.col("date").dt.month())
    .agg(
        # We name this 'third_thursday' to avoid the DuplicateError
        pl.col("date").sort().gather(2).alias("third_thursday") 
    )
    .sort("third_thursday")
    .explode("third_thursday")
    .get_column("third_thursday")
)

thursdays

third_thursday
date
2015-01-15
2015-02-19
2015-03-19
2015-04-16
2015-05-21
…
2015-08-20
2015-09-17
2015-10-15


* what the hail is this name collission error that i keep running into ??

* Pre-baked Logic: Pandas has a deep library of "Offest Aliases" like WOM-3THU that encode complex calendar rules directly into the freq parameter.

* Filter and Gather: Polars requires a "First Principles" approach: filter for the weekday, group by the month, and then gather (pick) the specific ordinal occurrence (the 3rd one).

* Monday-Sunday Indexing: Remember the 1-based vs 0-based difference. In Polars, Thursday is 4. In Pandas, Thursday is 3.

* The "Explode" Step: When aggregating in Polars, the result is often a list. Using .explode() flattens those lists back into individual rows.

* Name Collision: In Polars, the column used in group_by is automatically carried over to the output. If your aggregation (agg) also produces a column with that same name, Polars throws a DuplicateError.

* The Alias Fix: Using .alias("new_name") inside the aggregation is the standard way to prevent name conflicts.

* Property Access: In Polars, dt.month() is used to extract the month integer for grouping, while Pandas would typically use df.index.month.

* Explicit Ordinals: While Pandas uses hidden calendar logic (WOM-3THU), Polars makes the logic explicit: find all Thursdays, group them by month, and "gather" the 3rd one (index 2).

**38.** Some values in the the **FlightNumber** column are missing (they are `NaN`). These numbers are meant to increase by 10 with each row so 10055 and 10075 need to be put in place. Modify `df` to fill in these missing numbers and make the column an integer column (instead of a float column).

In [208]:
# ----- pandas
df = pd.DataFrame({'From_To': ['LoNDon_paris', 'MAdrid_miLAN', 'londON_StockhOlm', 
                               'Budapest_PaRis', 'Brussels_londOn'],
              'FlightNumber': [10045, np.nan, 10065, np.nan, 10085],
              'RecentDelays': [[23, 47], [], [24, 43, 87], [13], [67, 32]],
                   'Airline': ['KLM(!)', '<Air France> (12)', '(British Airways. )', 
                               '12. Air France', '"Swiss Air"']})
                
# 1. Interpolate the missing values
# 2. Convert to integer (interpolation results in floats)
df['FlightNumber'] = df['FlightNumber'].interpolate().astype(int)

print(df[['FlightNumber']])

   FlightNumber
0         10045
1         10055
2         10065
3         10075
4         10085


In [209]:
# ------ polars
pldf = pl.DataFrame({
    'From_To': ['LoNDon_paris', 'MAdrid_miLAN', 'londON_StockhOlm', 'Budapest_PaRis', 'Brussels_londOn'],
    'FlightNumber': [10045, None, 10065, None, 10085],
    'RecentDelays': [[23, 47], [], [24, 43, 87], [13], [67, 32]],
    'Airline': ['KLM(!)', '<Air France> (12)', '(British Airways. )', '12. Air France', '"Swiss Air"']
})

# 1. Cast to Float (interpolation requires floating point math)
# 2. Interpolate nulls
# 3. Cast back to Int64
result = pldf.with_columns(
    pl.col("FlightNumber").cast(pl.Float64).interpolate().cast(pl.Int64)
)

print(result.select("FlightNumber"))

shape: (5, 1)
┌──────────────┐
│ FlightNumber │
│ ---          │
│ i64          │
╞══════════════╡
│ 10045        │
│ 10055        │
│ 10065        │
│ 10075        │
│ 10085        │
└──────────────┘


* Linear Gaps: Both libraries use interpolate(), which assumes a linear progression between known points to fill the NaN or null values.

* Type Strictness: Polars requires numeric columns to be floats to perform interpolation math, whereas Pandas handles the transition from float (due to NaN) more implicitly.

* Integer Casting: Since missing value markers (NaN in Pandas) force columns to become floats, an explicit .astype(int) or .cast(pl.Int64) is required to return to whole numbers.

* Null Handling: Polars uses None for missing data, while Pandas traditionally uses np.nan (though it now supports a native pd.NA).

**39.** The **From\_To** column would be better as two separate columns! Split each string on the underscore delimiter `_` to give a new temporary DataFrame called 'temp' with the correct values. Assign the correct column names 'From' and 'To' to this temporary DataFrame. 

In [214]:
# ------Pandas 
# 1. Split the string on '_'
# 2. expand=True returns a DataFrame instead of a Series of lists
pd_temp = df['From_To'].str.split('_', expand=True)

# 3. Assign column names
pd_temp.columns = ['From', 'To']

pd_temp

,From,To
0,LoNDon,paris
1,MAdrid,miLAN
2,londON,StockhOlm
3,Budapest,PaRis
4,Brussels,londOn


In [215]:
# ------ Polars 
# 1. Split the string into a Struct with specific field names
# 2. Unnest the struct to turn those fields into top-level columns
pl_temp = (
    pldf.select(
        pl.col("From_To")
        .str.split_exact("_", 1)
        .struct.rename_fields(["From", "To"])
    )
    .unnest("From_To")
)

pl_temp

From,To
str,str
"""LoNDon""","""paris"""
"""MAdrid""","""miLAN"""
"""londON""","""StockhOlm"""
"""Budapest""","""PaRis"""
"""Brussels""","""londOn"""


* Expansion Logic: Pandas uses expand=True to jump directly from a Series to a DataFrame, while Polars uses a two-step process: creating a Struct and then unnest-ing it.

* Exact vs. General: Polars' split_exact is safer and faster because it forces you to specify how many splits you expect, preventing messy rows if an extra delimiter exists.

* Metadata Assignment: In Pandas, you rename columns by modifying the .columns attribute of the new object; in Polars, you define the names during the expression using rename_fields.

* Temporary Objects: Both methods result in a "Temporary" structure that is detached from the original DataFrame until you explicitly join or assign it back.

**40.** Notice how the capitalisation of the city names is all mixed up in this temporary DataFrame 'temp'. Standardise the strings so that only the first letter is uppercase (e.g. "londON" should become "London".)

In [216]:
# ------ pandas
# Apply capitalization to both columns
pd_temp['From'] = pd_temp['From'].str.capitalize()
pd_temp['To'] = pd_temp['To'].str.capitalize()

pd_temp

,From,To
0,London,Paris
1,Madrid,Milan
2,London,Stockholm
3,Budapest,Paris
4,Brussels,London


In [217]:
# ------ polars
# Apply to all columns in the 'temp' dataframe
pl_temp = pl_temp.with_columns(
    pl.all().str.to_titlecase()
)

pl_temp

From,To
str,str
"""London""","""Paris"""
"""Madrid""","""Milan"""
"""London""","""Stockholm"""
"""Budapest""","""Paris"""
"""Brussels""","""London"""


* Consistency: Both libraries provide dedicated string methods to handle "Sentence Case" (Capitalizing the first letter and lowercasing the rest).

* Column-Wise vs. Bulk: In Pandas, you typically update columns one by one; in Polars, you can use pl.all() or pl.col(pl.String) to standardize every string column in the DataFrame at once.

* Namespace: Both use a string-specific namespace (.str) to access these specialized text-processing functions.

* In-place vs. Functional: Pandas often involves direct assignment (temp['col'] = ...), while Polars follows a functional pattern (with_columns) that returns a new modified DataFrame.

**41.** Delete the From_To column from **41.** Delete the **From_To** column from `df` and attach the temporary DataFrame 'temp' from the previous questions.`df` and attach the temporary DataFrame from the previous questions.

In [ ]:
# ------ pandas
# 1. Drop the original 'From_To' column
df = df.drop('From_To', axis=1)

# 2. Concatenate the original df and the temp df along the columns axis (axis=1)
df = pd.concat([df, pd_temp], axis=1)
df

,FlightNumber,RecentDelays,Airline,From,To
0,10045,"[23, 47]",KLM(!),London,Paris
1,10055,[],<Air France> (12),Madrid,Milan
2,10065,"[24, 43, 87]",(British Airways. ),London,Stockholm
3,10075,[13],12. Air France,Budapest,Paris
4,10085,"[67, 32]","""Swiss Air""",Brussels,London


In [221]:
# ------ polars 
# 1. Drop 'From_To'
# 2. Use hstack to attach all columns from 'temp'
pldf = pldf.drop("From_To").hstack(pl_temp)

pldf

FlightNumber,RecentDelays,Airline,From,To
i64,list[i64],str,str,str
10045,"[23, 47]","""KLM(!)""","""London""","""Paris"""
null,[],"""<Air France> (12)""","""Madrid""","""Milan"""
10065,"[24, 43, 87]","""(British Airways. )""","""London""","""Stockholm"""
null,[13],"""12. Air France""","""Budapest""","""Paris"""
10085,"[67, 32]","""""Swiss Air""""","""Brussels""","""London"""


* Axis Alignment: In Pandas, axis=1 is the critical argument for both .drop() and pd.concat() to ensure you are operating on columns rather than rows.

* HStack vs Concat: Polars uses hstack (horizontal stack) as a convenient shorthand for gluing DataFrames together side-by-side without needing a complex library-level concat function.

* Side Effects: Pandas .drop() returns a copy by default; to modify the original without reassignment, you would need inplace=True (though reassignment is generally preferred).

* Selection Logic: Another common pattern in Polars is to simply select the columns you want to keep and add the new ones in a single chain, avoiding the "delete-then-attach" dance.

**42**. In the **Airline** column, you can see some extra puctuation and symbols have appeared around the airline names. Pull out just the airline name. E.g. `'(British Airways. )'` should become `'British Airways'`.

In [222]:
# ------ pandas 

# Regex breakdown:
# [a-zA-Z\s]+  -> Match one or more letters or whitespace characters
# ([a-zA-Z\s]+) -> Capture that group
df['Airline'] = df['Airline'].str.extract(r'([a-zA-Z\s]+)', expand=False).str.strip()

df['Airline']

0                KLM
1         Air France
2    British Airways
3         Air France
4          Swiss Air
Name: Airline, dtype: object

In [223]:
# ------ Polars
# Using extract with a similar regex pattern
# index=1 refers to the first capture group
result = pldf.with_columns(
    pl.col("Airline")
    .str.extract(r"([a-zA-Z\s]+)", 1)
    .str.strip_chars()
)

result.select("Airline")

Airline
str
"""KLM"""
"""Air France"""
"""British Airways"""
"""Air France"""
"""Swiss Air"""


* Capture Groups: Using parentheses () in a regex pattern allows you to "capture" the specific part of the string you want to keep while ignoring the rest.

* Character Classes: The pattern [a-zA-Z\s] specifically targets lowercase letters, uppercase letters, and whitespace, effectively filtering out punctuation and numbers.

* Greedy Matching: The + symbol ensures we grab the longest possible string of letters (e.g., "British Airways") rather than just the first letter.

* Cleaning the Clean: After extracting, it’s common to run a strip() (Pandas) or strip_chars() (Polars) to remove any lingering whitespace at the start or end of the captured string.

**43**. In the **RecentDelays** column, the values have been entered into the DataFrame as a list. We would like each first value in its own column, each second value in its own column, and so on. If there isn't an Nth value, the value should be NaN.

Expand the Series of lists into a new DataFrame named 'delays', rename the columns 'delay_1', 'delay_2', etc. and replace the unwanted RecentDelays column in `df` with 'delays'.

In [ ]:
# ------ Pandas

# 1. Expand the list column into a new DataFrame
delays = df['RecentDelays'].apply(pd.Series)

# 2. Rename columns using a list comprehension
delays.columns = [f'delay_{i+1}' for i in range(delays.shape[1])]

# 3. Drop the old column and join the new 'delays' DataFrame
df = df.drop('RecentDelays', axis=1).join(delays)

df

,FlightNumber,Airline,From,To,delay_1,delay_2,delay_3
0,10045,KLM,London,Paris,23.0,47.0,NaN
1,10055,Air France,Madrid,Milan,NaN,NaN,NaN
2,10065,British Airways,London,Stockholm,24.0,43.0,87.0
3,10075,Air France,Budapest,Paris,13.0,NaN,NaN
4,10085,Swiss Air,Brussels,London,67.0,32.0,NaN


In [226]:
# ------ Polars 
# 1. Determine the maximum length of the lists to set the "upper_bound"
max_width = pldf["RecentDelays"].list.len().max()

# 2. Convert to struct using the upper_bound and rename the fields
delays_pl = (
    pldf.select(
        pl.col("RecentDelays")
        .list.to_struct(upper_bound=max_width)
    )
    .unnest("RecentDelays")
)

# 3. Rename columns from "field_0", "field_1" to "delay_1", "delay_2"
delays_pl.columns = [f"delay_{i+1}" for i in range(len(delays_pl.columns))]

# 4. Attach back
pldf = pldf.drop("RecentDelays").hstack(delays_pl)

print(pldf)

shape: (5, 7)
┌──────────────┬─────────────────────┬──────────┬───────────┬─────────┬─────────┬─────────┐
│ FlightNumber ┆ Airline             ┆ From     ┆ To        ┆ delay_1 ┆ delay_2 ┆ delay_3 │
│ ---          ┆ ---                 ┆ ---      ┆ ---       ┆ ---     ┆ ---     ┆ ---     │
│ i64          ┆ str                 ┆ str      ┆ str       ┆ i64     ┆ i64     ┆ i64     │
╞══════════════╪═════════════════════╪══════════╪═══════════╪═════════╪═════════╪═════════╡
│ 10045        ┆ KLM(!)              ┆ London   ┆ Paris     ┆ 23      ┆ 47      ┆ null    │
│ null         ┆ <Air France> (12)   ┆ Madrid   ┆ Milan     ┆ null    ┆ null    ┆ null    │
│ 10065        ┆ (British Airways. ) ┆ London   ┆ Stockholm ┆ 24      ┆ 43      ┆ 87      │
│ null         ┆ 12. Air France      ┆ Budapest ┆ Paris     ┆ 13      ┆ null    ┆ null    │
│ 10085        ┆ "Swiss Air"         ┆ Brussels ┆ London    ┆ 67      ┆ 32      ┆ null    │
└──────────────┴─────────────────────┴──────────┴───────────┴─────

* Series-to-DataFrame: In Pandas, .apply(pd.Series) is a "magic" trick that treats each list like rows of a new DataFrame, automatically handling varying lengths with NaN.

* List-to-Struct: Polars uses the .list.to_struct() pattern. Since DataFrames can't have "variable length" columns, converting to a Struct (fixed fields) and then unnest-ing is the standard workflow.

* Dynamic Column Naming: Both solutions use dynamic naming (list comprehensions or lambda functions) because the number of columns depends on the length of the longest list in the data.

* Memory Efficiency: Polars' approach is highly optimized for memory; Pandas' .apply(pd.Series) can be very slow on large datasets because it essentially iterates through every row.

* Schema Enforcement: Unlike Pandas, which calculates the shape "on the fly," Polars prefers to know the column count upfront. Setting upper_bound tells Polars the maximum number of columns to create from the lists.

* Struct Fields: By default, .list.to_struct() names columns field_0, field_1, etc. If you want specific names like delay_1, you must rename them after the unnesting or by passing a list of names to the fields argument.

* Exploding vs. Expanding: Note that we are expanding (adding columns) rather than exploding (adding rows). Exploding would create a new row for every element in the list, while expanding keeps one row per original entry.

* Type Safety: This explicit approach prevents "unpredictable results" where different chunks of data might otherwise result in different numbers of columns during parallel processing.